# Qwen2.5-VL QLoRA: lấy best Chunk 2 rồi fine-tune lại Chunk 1 + xuất final adapter

Notebook này dùng cho trường hợp bạn đã train `chunk_02_02501_05000`, sau đó muốn lấy adapter/checkpoint tốt nhất của chunk 2 để **quay lại fine-tune trên chunk 1** (`00001–02500`) bằng collator đã sửa đúng label mask.

Output của lần train quay lại chunk 1 sẽ lưu ở:

```text
/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks/cycle_01_from_best_chunk02_train_00001_02500_adapter
```

Sau khi train xong, notebook sẽ tự động export thành **final adapter của toàn bộ quá trình fine-tune hiện tại** ở 2 dạng:

```text
/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks/final_finetuned_adapter
/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks/final_finetuned_adapter.zip
```

Lưu ý: `final_finetuned_adapter.zip` là file zip chứa đầy đủ LoRA adapter + tokenizer/processor config. Khi dùng inference hoặc train tiếp, giải nén zip hoặc dùng trực tiếp folder `final_finetuned_adapter`.


In [ ]:
# ============================================================
# 1. Cài thư viện
# Không cài lại torch để tránh lệch CUDA trên Colab.
# ============================================================
%pip -q install "transformers==4.51.3" "accelerate==1.6.0" "peft==0.15.2" "bitsandbytes==0.45.5" "qwen-vl-utils" "pillow" "tqdm" "pandas" "safetensors"


In [ ]:
# ============================================================
# 2. Import + kiểm tra GPU
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import json
import math
import shutil
import gc
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab hiện tại KHÔNG có GPU/CUDA.\n"
        "QLoRA 4-bit với bitsandbytes bắt buộc cần GPU NVIDIA.\n"
        "Vào Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU/L4/A100.\n"
        "Sau đó Restart runtime và chạy lại notebook."
    )

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")


In [ ]:
# ============================================================
# 3. Mount Drive + đọc dữ liệu
# Cấu trúc mong đợi:
# /content/drive/MyDrive/Final_Deeplearning/Split/train/train.jsonl
# /content/drive/MyDrive/Final_Deeplearning/Split/valid/valid.jsonl
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning/Split')
TRAIN_JSONL = DATA_ROOT / 'train' / 'train.jsonl'
VALID_JSONL = DATA_ROOT / 'valid' / 'valid.jsonl'

for path in [TRAIN_JSONL, VALID_JSONL]:
    assert path.exists(), f'Không tìm thấy file: {path}'


def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f'Lỗi JSONL dòng {line_no} ở {path}: {e}')
    return rows

train_rows = read_jsonl(TRAIN_JSONL)
valid_rows = read_jsonl(VALID_JSONL)

print('DATA_ROOT:', DATA_ROOT)
print('Train QA:', len(train_rows), 'Images:', len({x.get('image') for x in train_rows}))
print('Valid QA:', len(valid_rows), 'Images:', len({x.get('image') for x in valid_rows}))

# Nếu dataset của bạn đã đổi số dòng thì sửa hoặc comment assert này.
assert len(train_rows) == 23211, f'Tập train hiện tại có {len(train_rows)} dòng, không phải 23211. Nếu dataset đã đổi thì sửa assert này.'
train_rows[:2]


In [ ]:
# ============================================================
# 4. CONFIG: dùng best chunk 2 để train lại chunk 1
# ============================================================

# Folder chính chứa chunk_01, chunk_02, ...
SEQ_OUTPUT_DIR = Path('/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks')
SEQ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chunk nguồn: đã train xong chunk 2.
SOURCE_CHUNK_ID = 2
SOURCE_CHUNK_NAME = 'chunk_02_02501_05000'
SOURCE_WORK_DIR = SEQ_OUTPUT_DIR / f'{SOURCE_CHUNK_NAME}_work'
SOURCE_FINAL_ADAPTER_DIR = SEQ_OUTPUT_DIR / f'{SOURCE_CHUNK_NAME}_adapter'

# Chunk đích: quay lại train lại chunk 1.
TARGET_CHUNK_ID = 1
CHUNK_SIZE = 2500
START_INDEX = 0
END_INDEX = min(CHUNK_SIZE, len(train_rows))

train_rows_chunk = train_rows[START_INDEX:END_INDEX]

# Eval subset để chọn best checkpoint trong lần train quay lại chunk 1.
MAX_VALID_SAMPLES = 500
valid_rows_eval = valid_rows[:MAX_VALID_SAMPLES] if MAX_VALID_SAMPLES else valid_rows

# Không ghi đè chunk_01 cũ. Lưu ra một tên mới.
OUTPUT_TAG = 'cycle_01_from_best_chunk02_train_00001_02500'
WORK_DIR = SEQ_OUTPUT_DIR / f'{OUTPUT_TAG}_work'
ADAPTER_OUTPUT_DIR = SEQ_OUTPUT_DIR / f'{OUTPUT_TAG}_adapter'
SUMMARY_PATH = ADAPTER_OUTPUT_DIR / 'chunk_summary.json'

# Final adapter của toàn bộ quá trình fine-tune hiện tại.
# Đây là folder/zip cuối cùng bạn dùng để inference hoặc train tiếp.
FINAL_ADAPTER_DIR = SEQ_OUTPUT_DIR / 'final_finetuned_adapter'
FINAL_ZIP_BASENAME = SEQ_OUTPUT_DIR / 'final_finetuned_adapter'
FINAL_ZIP_PATH = SEQ_OUTPUT_DIR / 'final_finetuned_adapter.zip'

TRAIN_LR = 1e-5
NUM_TRAIN_EPOCHS = 1
GRAD_ACCUM_STEPS = 8
LOGGING_STEPS = 20
EVAL_STEPS = 50
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

print('SEQ_OUTPUT_DIR:', SEQ_OUTPUT_DIR)
print('SOURCE_WORK_DIR:', SOURCE_WORK_DIR)
print('SOURCE_FINAL_ADAPTER_DIR:', SOURCE_FINAL_ADAPTER_DIR)
print('TARGET TRAIN samples 1-based:', START_INDEX + 1, '->', END_INDEX)
print('Num train samples:', len(train_rows_chunk))
print('Num valid eval samples:', len(valid_rows_eval))
print('WORK_DIR:', WORK_DIR)
print('ADAPTER_OUTPUT_DIR:', ADAPTER_OUTPUT_DIR)
print('FINAL_ADAPTER_DIR:', FINAL_ADAPTER_DIR)
print('FINAL_ZIP_PATH:', FINAL_ZIP_PATH)

assert len(train_rows_chunk) > 0, 'Chunk 1 rỗng, kiểm tra lại dữ liệu train.'


In [ ]:
# ============================================================
# 5. Chọn adapter/checkpoint tốt nhất của chunk 2
# ============================================================

def adapter_files_exist(path: Path) -> bool:
    path = Path(path)
    return (
        (path / 'adapter_config.json').exists()
        and ((path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists())
    )


def checkpoint_step(path: Path):
    m = re.search(r'checkpoint-(\d+)$', str(path))
    return int(m.group(1)) if m else None


def sorted_checkpoints(work_dir: Path):
    work_dir = Path(work_dir)
    ckpts = []
    if work_dir.exists():
        for p in work_dir.glob('checkpoint-*'):
            if p.is_dir():
                step = checkpoint_step(p)
                if step is not None:
                    ckpts.append((step, p))
    return [p for _, p in sorted(ckpts, key=lambda x: x[0])]


def read_json_safe(path: Path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None


def resolve_best_model_checkpoint_from_state(work_dir: Path):
    ckpts = sorted_checkpoints(work_dir)
    for ckpt in reversed(ckpts):
        state = read_json_safe(ckpt / 'trainer_state.json')
        if not state:
            continue
        best = state.get('best_model_checkpoint')
        if not best:
            continue
        best_path = Path(best)
        if not best_path.exists():
            # Nếu path tuyệt đối trong trainer_state không còn khớp, thử lấy basename trong work_dir.
            best_path = work_dir / Path(best).name
        if adapter_files_exist(best_path):
            return best_path, state.get('best_metric'), 'trainer_state.best_model_checkpoint'
    return None, None, None


def resolve_lowest_eval_checkpoint(work_dir: Path):
    ckpts = sorted_checkpoints(work_dir)
    if not ckpts:
        return None, None, None

    # Ưu tiên đọc log_history từ checkpoint mới nhất, vì nó thường chứa toàn bộ lịch sử train.
    latest_state = read_json_safe(ckpts[-1] / 'trainer_state.json')
    if latest_state and isinstance(latest_state.get('log_history'), list):
        best_step = None
        best_loss = None
        for item in latest_state['log_history']:
            if 'eval_loss' in item and 'step' in item:
                try:
                    loss = float(item['eval_loss'])
                    step = int(item['step'])
                except Exception:
                    continue
                if best_loss is None or loss < best_loss:
                    best_loss = loss
                    best_step = step
        if best_step is not None:
            best_ckpt = work_dir / f'checkpoint-{best_step}'
            if adapter_files_exist(best_ckpt):
                return best_ckpt, best_loss, 'lowest eval_loss from latest trainer_state.log_history'

    # Fallback: đọc từng checkpoint, tìm eval_loss tại đúng step checkpoint.
    best_ckpt = None
    best_loss = None
    for ckpt in ckpts:
        step = checkpoint_step(ckpt)
        state = read_json_safe(ckpt / 'trainer_state.json')
        if not state or not isinstance(state.get('log_history'), list):
            continue
        for item in state['log_history']:
            if item.get('step') == step and 'eval_loss' in item:
                try:
                    loss = float(item['eval_loss'])
                except Exception:
                    continue
                if adapter_files_exist(ckpt) and (best_loss is None or loss < best_loss):
                    best_ckpt = ckpt
                    best_loss = loss
    if best_ckpt is not None:
        return best_ckpt, best_loss, 'lowest eval_loss from checkpoint trainer_state'

    return None, None, None


def resolve_best_chunk2_adapter():
    # 1) Nếu chunk 2 có trainer_state.best_model_checkpoint thì dùng đúng best checkpoint.
    p, metric, reason = resolve_best_model_checkpoint_from_state(SOURCE_WORK_DIR)
    if p is not None:
        return p, metric, reason

    # 2) Nếu có eval_loss trong log_history thì chọn checkpoint có eval_loss thấp nhất.
    p, metric, reason = resolve_lowest_eval_checkpoint(SOURCE_WORK_DIR)
    if p is not None:
        return p, metric, reason

    # 3) Nếu không có eval metrics, dùng adapter final của chunk 2.
    if adapter_files_exist(SOURCE_FINAL_ADAPTER_DIR):
        metric = None
        summary = read_json_safe(SOURCE_FINAL_ADAPTER_DIR / 'chunk_summary.json')
        if summary and summary.get('eval_loss') is not None:
            try:
                metric = float(summary.get('eval_loss'))
            except Exception:
                metric = None
        return SOURCE_FINAL_ADAPTER_DIR, metric, 'final chunk_02 adapter; no checkpoint eval metric found'

    # 4) Fallback cuối: latest checkpoint có adapter file.
    ckpts = sorted_checkpoints(SOURCE_WORK_DIR)
    for ckpt in reversed(ckpts):
        if adapter_files_exist(ckpt):
            return ckpt, None, 'latest checkpoint fallback; no eval metric found'

    raise FileNotFoundError(
        'Không tìm thấy adapter/checkpoint usable của chunk 2. Cần có một trong các folder sau:\n'
        f'- {SOURCE_FINAL_ADAPTER_DIR}\n'
        f'- {SOURCE_WORK_DIR}/checkpoint-*\n'
        'Trong đó phải có adapter_config.json và adapter_model.safetensors hoặc adapter_model.bin.'
    )

SOURCE_ADAPTER_DIR, SOURCE_BEST_METRIC, SOURCE_REASON = resolve_best_chunk2_adapter()

print('=' * 80)
print('[SOURCE] Adapter/checkpoint sẽ dùng để quay lại train chunk 1')
print('=' * 80)
print('SOURCE_ADAPTER_DIR:', SOURCE_ADAPTER_DIR)
print('SOURCE_BEST_METRIC:', SOURCE_BEST_METRIC)
print('SOURCE_REASON:', SOURCE_REASON)
print('adapter_config.json:', (SOURCE_ADAPTER_DIR / 'adapter_config.json').exists())
print('adapter_model.safetensors:', (SOURCE_ADAPTER_DIR / 'adapter_model.safetensors').exists())
print('adapter_model.bin:', (SOURCE_ADAPTER_DIR / 'adapter_model.bin').exists())


In [ ]:
# ============================================================
# 6. Load Qwen2.5-VL 3B 4-bit + load best adapter chunk 2 để train tiếp
# ============================================================
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Giảm vision tokens để tránh OOM trên T4.
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False

# Freeze base model; chỉ LoRA adapter được train.
for param in base_model.parameters():
    param.requires_grad = False

if hasattr(base_model, 'gradient_checkpointing_enable'):
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

if hasattr(base_model, 'enable_input_require_grads'):
    base_model.enable_input_require_grads()

print('Load best chunk 2 adapter:')
print(SOURCE_ADAPTER_DIR)

model = PeftModel.from_pretrained(
    base_model,
    str(SOURCE_ADAPTER_DIR),
    is_trainable=True,
)

print('\nTrainable parameters:')
model.print_trainable_parameters()

free, total = torch.cuda.mem_get_info()
print(f'Free VRAM after load: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB')


In [ ]:
# ============================================================
# 7. Prompt, resolve đường dẫn ảnh, Dataset, collator ĐÚNG MASK LABEL
# ============================================================
from torch.utils.data import Dataset


def image_root_for_split(split):
    return DATA_ROOT / split


def normalize_image_relpath(x):
    return str(x).strip().replace('\\', '/').lstrip('/')


def resolve_image_path(row, split=None):
    raw = normalize_image_relpath(row['image'])
    p = Path(raw)
    candidates = []

    if p.is_absolute():
        candidates.append(p)

    split_candidates = []
    if split:
        split_candidates.append(split)

    for s in ['train', 'valid', 'val', 'test']:
        if s not in split_candidates:
            split_candidates.append(s)

    for s in split_candidates:
        split_root = image_root_for_split(s)
        candidates.append(split_root / raw)                 # images/Pxxxx.png
        candidates.append(split_root / 'images' / p.name)   # Pxxxx.png

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        'Không tìm thấy ảnh.\n'
        f"id: {row.get('id')}\n"
        f"split truyền vào: {split}\n"
        f"image trong JSONL: {row.get('image')}\n"
        'Đã thử:\n' + '\n'.join(str(x) for x in candidates[:20])
    )


def build_user_text(question):
    return (
        'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
        'Hãy trả lời câu hỏi bằng tiếng Việt, thật ngắn gọn, tối đa 10 từ. '
        'Không giải thích thêm.\n'
        f'Câu hỏi: {question}'
    )


def make_messages(row, include_answer=True):
    split = row.get('__split', None)
    image_path = resolve_image_path(row, split=split)

    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'image': str(image_path),
                    'min_pixels': MIN_PIXELS,
                    'max_pixels': MAX_PIXELS,
                },
                {'type': 'text', 'text': build_user_text(row['question'])},
            ],
        }
    ]

    if include_answer:
        messages.append({
            'role': 'assistant',
            'content': [{'type': 'text', 'text': str(row['answer'])}],
        })

    return messages


class HerbVQADataset(Dataset):
    def __init__(self, rows, split):
        self.rows = rows
        self.split = split

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = dict(self.rows[idx])
        row['__split'] = self.split
        return row


def collate_fn(batch):
    """
    Collator đúng cho Qwen2.5-VL SFT/VQA:
    - Full input: image + question + assistant answer.
    - Labels: chỉ tính loss trên assistant answer.
    - Prompt/image/question đều vẫn nằm trong input, nhưng label = -100 để không tính loss.
    """
    processor.tokenizer.padding_side = 'right'

    full_messages = [make_messages(row, include_answer=True) for row in batch]
    prompt_messages = [make_messages(row, include_answer=False) for row in batch]

    full_texts = [
        processor.apply_chat_template(
            m,
            tokenize=False,
            add_generation_prompt=False,
        )
        for m in full_messages
    ]

    prompt_texts = [
        processor.apply_chat_template(
            m,
            tokenize=False,
            add_generation_prompt=True,
        )
        for m in prompt_messages
    ]

    full_image_inputs, full_video_inputs = process_vision_info(full_messages)
    inputs = processor(
        text=full_texts,
        images=full_image_inputs,
        videos=full_video_inputs,
        padding=True,
        return_tensors='pt',
    )

    labels = inputs['input_ids'].clone()

    # Mask padding.
    labels[inputs['attention_mask'] == 0] = -100

    # Tính prompt_len bằng processor kèm ảnh, không dùng tokenizer text thường.
    prompt_image_inputs, prompt_video_inputs = process_vision_info(prompt_messages)
    prompt_inputs = processor(
        text=prompt_texts,
        images=prompt_image_inputs,
        videos=prompt_video_inputs,
        padding=True,
        return_tensors='pt',
    )

    for i in range(len(batch)):
        prompt_len = int(prompt_inputs['attention_mask'][i].sum().item())
        labels[i, :prompt_len] = -100

    inputs['labels'] = labels
    return inputs


def check_missing_images(dataset, max_show=10):
    missing = []
    for i in range(len(dataset)):
        row = dataset[i]
        split = row.get('__split', getattr(dataset, 'split', None))
        try:
            _ = resolve_image_path(row, split=split)
        except Exception as e:
            missing.append((i, row.get('id'), split, row.get('image'), str(e)))

    print('Tổng samples:', len(dataset))
    print('Số ảnh lỗi:', len(missing))
    for item in missing[:max_show]:
        print(item)
    return missing


train_dataset = HerbVQADataset(train_rows_chunk, 'train')
valid_dataset = HerbVQADataset(valid_rows_eval, 'valid')

missing_train = check_missing_images(train_dataset)
missing_valid = check_missing_images(valid_dataset)

if len(missing_train) > 0 or len(missing_valid) > 0:
    raise FileNotFoundError('Vẫn còn ảnh lỗi đường dẫn. Xem danh sách ở trên.')


In [ ]:
# ============================================================
# 8. Check label mask: đảm bảo model chỉ học ANSWER
# ============================================================

def inspect_label_mask(dataset, sample_idx=0):
    sample = dataset[sample_idx]
    batch = collate_fn([sample])

    input_ids = batch['input_ids'][0]
    labels = batch['labels'][0]

    total_tokens = labels.numel()
    ignored_tokens = int((labels == -100).sum().item())
    learn_tokens = int((labels != -100).sum().item())
    learn_ids = input_ids[labels != -100]

    print('=' * 80)
    print('CHECK LABEL MASK')
    print('=' * 80)
    print('Sample idx        :', sample_idx)
    print('Input shape       :', tuple(batch['input_ids'].shape))
    print('Total tokens      :', total_tokens)
    print('Ignored -100      :', ignored_tokens)
    print('Learn tokens      :', learn_tokens)
    print('Learn token ratio :', round(learn_tokens / max(total_tokens, 1), 6))

    print('\n[TEXT MODEL ĐANG HỌC - RAW]')
    print(processor.tokenizer.decode(learn_ids, skip_special_tokens=False))

    print('\n[TEXT MODEL ĐANG HỌC - CLEAN]')
    print(processor.tokenizer.decode(learn_ids, skip_special_tokens=True))

    print('\n[ANSWER GỐC]')
    print(sample.get('answer'))

    if learn_tokens <= 0:
        raise RuntimeError('Không có token nào để học. labels toàn -100.')
    if learn_tokens / max(total_tokens, 1) > 0.3:
        print('[WARNING] Learn token ratio hơi cao. Hãy xem TEXT MODEL ĐANG HỌC có bị dính prompt không.')
    else:
        print('[OK] Label mask có vẻ đúng.')

inspect_label_mask(train_dataset, sample_idx=0)


In [ ]:
# ============================================================
# 9. Eval nhanh trước khi train lại chunk 1
# ============================================================
from transformers import TrainingArguments, Trainer

pre_eval_args = TrainingArguments(
    output_dir=str(WORK_DIR / 'pre_eval_tmp'),
    per_device_eval_batch_size=1,
    remove_unused_columns=False,
    report_to='none',
    fp16=True,
    label_names=['labels'],
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

pre_eval_trainer = Trainer(
    model=model,
    args=pre_eval_args,
    eval_dataset=valid_dataset,
    data_collator=collate_fn,
)

print('=' * 80)
print('[PRE-EVAL] Eval trước khi train lại chunk 1')
print('=' * 80)
pre_metrics = pre_eval_trainer.evaluate()
print(pre_metrics)

pre_loss = float(pre_metrics.get('eval_loss', 999999.0))
print('pre_eval_loss:', pre_loss)
print('pre_eval_perplexity:', math.exp(min(pre_loss, 20)))

del pre_eval_trainer

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 10. Train lại chunk 1 từ best chunk 2 + lưu best adapter
# ============================================================
from transformers import TrainingArguments, Trainer


def find_last_checkpoint(output_dir):
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return None

    checkpoints = []
    for p in output_dir.glob('checkpoint-*'):
        if p.is_dir():
            step = checkpoint_step(p)
            if step is not None:
                checkpoints.append((step, p))

    if not checkpoints:
        return None

    return str(sorted(checkpoints, key=lambda x: x[0])[-1][1])


def copy_tree_clean(src_dir: Path, dst_dir: Path):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)

    if not adapter_files_exist(src_dir):
        raise FileNotFoundError(
            'Không thể export final adapter vì source adapter không đủ file.\n'
            f'Source: {src_dir}\n'
            'Cần có adapter_config.json và adapter_model.safetensors hoặc adapter_model.bin.'
        )

    if dst_dir.exists():
        shutil.rmtree(dst_dir)

    shutil.copytree(src_dir, dst_dir)
    return dst_dir


def export_final_adapter(src_adapter_dir: Path, extra_summary: dict | None = None):
    """
    Export adapter cuối cùng của quá trình fine-tune hiện tại.
    - FINAL_ADAPTER_DIR: folder dùng trực tiếp cho inference/train tiếp.
    - FINAL_ZIP_PATH: một file zip để tải về/lưu trữ.
    """
    src_adapter_dir = Path(src_adapter_dir)

    print('\n' + '=' * 80)
    print('[EXPORT FINAL ADAPTER]')
    print('=' * 80)
    print('Source adapter:', src_adapter_dir)
    print('Final adapter folder:', FINAL_ADAPTER_DIR)
    print('Final adapter zip:', FINAL_ZIP_PATH)

    copy_tree_clean(src_adapter_dir, FINAL_ADAPTER_DIR)

    final_summary = {
        'export_type': 'final_finetuned_lora_adapter',
        'note': 'Đây là final LoRA adapter sau quy trình: best chunk 2 -> train lại chunk 1.',
        'source_adapter_dir': str(src_adapter_dir),
        'final_adapter_dir': str(FINAL_ADAPTER_DIR),
        'final_zip_path': str(FINAL_ZIP_PATH),
        'created_at': datetime.now().isoformat(),
    }
    if extra_summary:
        final_summary.update(extra_summary)

    with open(FINAL_ADAPTER_DIR / 'final_adapter_summary.json', 'w', encoding='utf-8') as f:
        json.dump(final_summary, f, ensure_ascii=False, indent=2)

    # Xóa zip cũ nếu có, rồi tạo zip mới.
    if FINAL_ZIP_PATH.exists():
        FINAL_ZIP_PATH.unlink()

    shutil.make_archive(
        base_name=str(FINAL_ZIP_BASENAME),
        format='zip',
        root_dir=str(FINAL_ADAPTER_DIR.parent),
        base_dir=FINAL_ADAPTER_DIR.name,
    )

    if not FINAL_ZIP_PATH.exists():
        raise FileNotFoundError(f'Tạo zip thất bại: {FINAL_ZIP_PATH}')

    print('[OK] Đã export final adapter folder:', FINAL_ADAPTER_DIR)
    print('[OK] Đã tạo final adapter zip:', FINAL_ZIP_PATH)
    print('Zip size MB:', round(FINAL_ZIP_PATH.stat().st_size / 1024**2, 2))

    return FINAL_ADAPTER_DIR, FINAL_ZIP_PATH


adapter_exists = adapter_files_exist(ADAPTER_OUTPUT_DIR)
summary_exists = SUMMARY_PATH.exists()

if adapter_exists and summary_exists:
    print('Adapter output đã tồn tại và có chunk_summary.json, bỏ qua train lại:')
    print(ADAPTER_OUTPUT_DIR)

    old_summary = read_json_safe(SUMMARY_PATH) or {}
    export_final_adapter(
        ADAPTER_OUTPUT_DIR,
        extra_summary={
            'train_was_skipped': True,
            'existing_summary': old_summary,
        }
    )
else:
    if adapter_exists and not summary_exists:
        raise RuntimeError(
            'Adapter output đã tồn tại nhưng thiếu chunk_summary.json.\n'
            'Để tránh ghi đè sai trạng thái, hãy xóa folder adapter output rồi chạy lại:\n'
            f'{ADAPTER_OUTPUT_DIR}'
        )

    training_args = TrainingArguments(
        output_dir=str(WORK_DIR),

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,

        learning_rate=TRAIN_LR,
        num_train_epochs=NUM_TRAIN_EPOCHS,

        logging_steps=LOGGING_STEPS,

        # Có eval để chọn best checkpoint thật sự cho lần quay lại chunk 1.
        eval_strategy='steps',
        eval_steps=EVAL_STEPS,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,

        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,

        fp16=True,
        bf16=False,

        remove_unused_columns=False,
        report_to='none',

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},

        optim='paged_adamw_8bit',
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        label_names=['labels'],

        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=collate_fn,
    )

    resume_ckpt = find_last_checkpoint(WORK_DIR)

    print('=' * 80)
    print('[TRAIN] Bắt đầu train lại chunk 1 từ best chunk 2')
    print('=' * 80)
    print('SOURCE_ADAPTER_DIR:', SOURCE_ADAPTER_DIR)
    print('SOURCE_REASON:', SOURCE_REASON)
    print('SOURCE_BEST_METRIC:', SOURCE_BEST_METRIC)
    print('WORK_DIR:', WORK_DIR)
    print('ADAPTER_OUTPUT_DIR:', ADAPTER_OUTPUT_DIR)
    print('Resume checkpoint:', resume_ckpt)
    print('Learning rate:', TRAIN_LR)
    print('Train samples:', len(train_dataset))
    print('Valid samples:', len(valid_dataset))
    print('Gradient accumulation:', GRAD_ACCUM_STEPS)
    print('Effective batch size:', GRAD_ACCUM_STEPS)

    if resume_ckpt:
        train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    else:
        train_result = trainer.train()

    print('\n[EVAL] Eval sau khi train. Vì load_best_model_at_end=True, model hiện tại là checkpoint tốt nhất theo eval_loss.')
    eval_metrics = trainer.evaluate()
    eval_loss = float(eval_metrics.get('eval_loss', 999999.0))
    eval_ppl = math.exp(min(eval_loss, 20))

    ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(ADAPTER_OUTPUT_DIR))
    processor.save_pretrained(str(ADAPTER_OUTPUT_DIR))

    summary = {
        'train_mode': 'cycle_back_chunk1_from_best_chunk2',
        'source_chunk_id': SOURCE_CHUNK_ID,
        'source_chunk_name': SOURCE_CHUNK_NAME,
        'source_adapter_dir': str(SOURCE_ADAPTER_DIR),
        'source_best_metric': SOURCE_BEST_METRIC,
        'source_reason': SOURCE_REASON,

        'target_chunk_id': TARGET_CHUNK_ID,
        'target_train_start_index_0_based': START_INDEX,
        'target_train_end_index_0_based_exclusive': END_INDEX,
        'target_train_sample_from_1_based': START_INDEX + 1,
        'target_train_sample_to_1_based': END_INDEX,

        'num_train_samples': len(train_dataset),
        'num_valid_samples': len(valid_dataset),
        'learning_rate': TRAIN_LR,
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'gradient_accumulation_steps': GRAD_ACCUM_STEPS,

        'pre_eval_loss': pre_loss,
        'final_eval_loss': eval_loss,
        'final_eval_perplexity': eval_ppl,
        'eval_metrics': eval_metrics,

        'adapter_dir': str(ADAPTER_OUTPUT_DIR),
        'work_dir': str(WORK_DIR),
        'model_id': MODEL_ID,
        'finished_at': datetime.now().isoformat(),
    }

    with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # Export thành final adapter của toàn bộ quá trình fine-tune hiện tại.
    export_final_adapter(
        ADAPTER_OUTPUT_DIR,
        extra_summary={
            'train_was_skipped': False,
            'cycle_summary': summary,
        }
    )

    print('\n' + '=' * 80)
    print('[DONE] Đã lưu adapter tốt nhất sau khi quay lại train chunk 1')
    print('=' * 80)
    print('Saved adapter:', ADAPTER_OUTPUT_DIR)
    print('Saved summary:', SUMMARY_PATH)
    print('Final eval_loss:', eval_loss)
    print('Final eval_perplexity:', eval_ppl)

    del trainer
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 11. Kiểm tra file output cuối cùng
# ============================================================
print('ADAPTER_OUTPUT_DIR:', ADAPTER_OUTPUT_DIR)
print('FINAL_ADAPTER_DIR:', FINAL_ADAPTER_DIR)
print('FINAL_ZIP_PATH:', FINAL_ZIP_PATH)

print('\n' + '=' * 80)
print('[CHECK] Adapter của lần quay lại train chunk 1')
print('=' * 80)
for name in [
    'adapter_config.json',
    'adapter_model.safetensors',
    'adapter_model.bin',
    'preprocessor_config.json',
    'tokenizer_config.json',
    'chunk_summary.json',
]:
    p = ADAPTER_OUTPUT_DIR / name
    print(f'{name}:', p.exists(), '->', p)

print('\n' + '=' * 80)
print('[CHECK] Final adapter folder/zip')
print('=' * 80)
for name in [
    'adapter_config.json',
    'adapter_model.safetensors',
    'adapter_model.bin',
    'preprocessor_config.json',
    'tokenizer_config.json',
    'chunk_summary.json',
    'final_adapter_summary.json',
]:
    p = FINAL_ADAPTER_DIR / name
    print(f'{name}:', p.exists(), '->', p)

print('final_finetuned_adapter.zip:', FINAL_ZIP_PATH.exists(), '->', FINAL_ZIP_PATH)
if FINAL_ZIP_PATH.exists():
    print('Zip size MB:', round(FINAL_ZIP_PATH.stat().st_size / 1024**2, 2))

if SUMMARY_PATH.exists():
    with open(SUMMARY_PATH, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print('\nSUMMARY cycle adapter:')
    print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

final_summary_path = FINAL_ADAPTER_DIR / 'final_adapter_summary.json'
if final_summary_path.exists():
    with open(final_summary_path, 'r', encoding='utf-8') as f:
        final_summary = json.load(f)
    print('\nSUMMARY final adapter:')
    print(json.dumps(final_summary, ensure_ascii=False, indent=2)[:4000])
